In [5]:
"""1) Decision Variables:
    x = units of Product A (per week)
    y = Units of Product B (per week)
    z = 1 if Machine 2 goes into overtime (Saturday), else 0

2) Parameters:
    ?

3) Constraints:
    Machine 1:
        2x + 4y <= 100

    Machine 2:
        3x + 2y <= 80 + 20z

    B_min: y >= 10
    Nonegativity/Binary:
        x ≥ 0
        y ≥ 0
        z ∈ (0, 1)
"""
import pulp as pl
from textwrap import dedent

def build_model(model_name):
    """
    Maximization model for weekly profit, subject to resources and time constraints.
    ----------------------------------------------------------v-
    INPUT:
        model_name: (str) Name of model

    OUTPUT:
        model: (pulp.LpProblem) 
    """
    # Input validation
    if not isinstance(model_name, str):
        raise TypeError("\nInput type invalid")

    return pl.LpProblem(model_name, sense=pl.LpMaximize)

# Build optimization model
model = build_model("two_products")


In [6]:
#  Decision variables
x = pl.LpVariable(name="Product_A_units", lowBound=0, cat="Continuous")
y = pl.LpVariable(name="Product_B_units", lowBound=0, cat="Continuous")
z = pl.LpVariable(name="overtime", cat=pl.LpBinary)

# Objective function (weekly proft)
# Penalty of $500 only when z=1 (overtime)
model += 40 * x + 60 * y - 500 * z, "Weekly_profit"


In [7]:
# Adding constraints
# Machine 1
model += 2 * x + 4 * y <= 100, "machine_1_capacity"

# Machine 2
model += 3 * x + 2 * y <= 80 + 20 * z, "machine_2_capacity"

# Min production for B
model += y >= 10, "minimum_B_quota"

# Solving model
solver = pl.PULP_CBC_CMD(msg=False)
model.solve(solver)

# Solution
stat = pl.LpStatus[model.status]
opt_x = pl.value(x)
opt_y = pl.value(y)
opt_z = pl.value(z)
optimal = pl.value(model.objective)

# Output
print(f"\nSolver status: {stat}")
print(f"x = units of Product A = {opt_x:.3f}")
print(f"y = units of Product B = {opt_y:.3f}")
print(f"Overtime decision = {int(round(opt_z))}")

print("\nObjective Value: (maximum weekly profit)")
print(f"${optimal:.2f}")



Solver status: Optimal
x = units of Product A = 15.000
y = units of Product B = 17.500
Overtime decision = 0

Objective Value: (maximum weekly profit)
$1650.00


In [8]:
# Interpretation
print(dedent(f"""
INTERPRETATION
----------------
The optimal weekly profit is achieved via {opt_x:.2f} units of
Product A and {opt_y:.2f} units of Product B.

Thus, we get a maximum weekly profit of ${optimal:.2f}.

Since z = 0, the model suggests we don't run machine 2 on a Saturday (now overtime).
      """))



INTERPRETATION
----------------
The optimal weekly profit is achieved via 15.00 units of
Product A and 17.50 units of Product B.

Thus, we get a maximum weekly profit of $1650.00.

Since z = 0, the model suggests we don't run machine 2 on a Saturday (now overtime).

